# Metadata Analysis (breaking down XLSX files into XML docs)

In [ ]:
# @title
import os
import zipfile
import shutil
import difflib

# --- CONFIGURATION ---
GOOD_FILE = "good.xlsx" # Your known-good reference file
BAD_FILE = "FIXED_SURGICALLY.xlsx"        # The file that fails
# --- END CONFIGURATION ---

# Define temporary directories
GOOD_DIR = "good_file_unzipped"
BAD_DIR = "bad_file_unzipped"

def unzip_file(filepath, out_dir):
    """Unzips an .xlsx file to a specified directory."""
    print(f"\nExtracting {filepath} to {out_dir}...")

    # Clean up old directory if it exists
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)

    try:
        with zipfile.ZipFile(filepath, 'r') as zip_ref:
            zip_ref.extractall(out_dir)
        print("  > Success.")
        return True
    except zipfile.BadZipFile:
        print(f"  ❌ ERROR: File is corrupt and cannot be unzipped: {filepath}")
        return False
    except Exception as e:
        print(f"  ❌ ERROR: Could not unzip file. Error: {e}")
        return False

def get_all_files(directory):
    """Gets a set of all relative file paths from the base directory."""
    file_set = set()
    for root, _, files in os.walk(directory):
        for file in files:
            full_path = os.path.join(root, file)
            # Get relative path from the base directory
            rel_path = os.path.relpath(full_path, directory)
            # Use forward slashes for cross-platform consistency
            file_set.add(rel_path.replace("\\", "/"))
    return file_set

def compare_file_lists(good_files, bad_files):
    """Compares the two sets of files and prints differences."""
    print("\n--- 1. Comparing File Lists ---")

    missing_from_bad = good_files - bad_files
    added_in_bad = bad_files - good_files

    if not missing_from_bad and not added_in_bad:
        print("  ✅ OK: File lists match perfectly.")
        return

    if missing_from_bad:
        print("  ⚠️ Files in GOOD file but MISSING from BAD file:")
        for f in sorted(missing_from_bad):
            print(f"    - {f}")

    if added_in_bad:
        print("  ❌ Files in BAD file but NOT in GOOD file (High-value clues!):")
        for f in sorted(added_in_bad):
            print(f"    - {f}")

def compare_file_properties(good_dir, bad_dir, common_files):
    """Compares file sizes for all common files."""
    print("\n--- 2. Comparing File Sizes ---")
    print("  (Looking for major size differences, e.g., in styles.xml)")

    warnings = []
    for file_rel_path in sorted(common_files):
        good_path = os.path.join(good_dir, file_rel_path)
        bad_path = os.path.join(bad_dir, file_rel_path)

        try:
            good_size = os.path.getsize(good_path)
            bad_size = os.path.getsize(bad_path)

            if good_size != bad_size:
                size_diff = bad_size - good_size
                percent_diff = (size_diff / good_size * 100) if good_size > 0 else float('inf')

                # Only flag significant differences
                if abs(size_diff) > 1024 or abs(percent_diff) > 50:
                    warnings.append(
                        f"  ⚠️ {file_rel_path}: "
                        f"GOOD={good_size} bytes, BAD={bad_size} bytes "
                        f"(Diff: {size_diff:+} bytes / {percent_diff:+.1f}%)"
                    )
        except FileNotFoundError:
            continue

    if not warnings:
        print("  ✅ OK: No significant file size differences found.")
    else:
        for w in warnings:
            print(w)

def compare_file_content(good_dir, bad_dir, file_rel_path):
    """Performs a text diff on two files and prints the changes."""
    print(f"\n--- 3. Content Diff for: {file_rel_path} ---")

    good_path = os.path.join(good_dir, file_rel_path)
    bad_path = os.path.join(bad_dir, file_rel_path)

    if not os.path.exists(good_path):
        print(f"  > Skipping: Not in good file.")
        return
    if not os.path.exists(bad_path):
        print(f"  > Skipping: Not in bad file.")
        return

    try:
        with open(good_path, 'r', encoding='utf-8') as f_good:
            good_lines = f_good.readlines()
        with open(bad_path, 'r', encoding='utf-8') as f_bad:
            bad_lines = f_bad.readlines()
    except UnicodeDecodeError:
        print("  > Skipping: File is binary and cannot be text-diffed.")
        return
    except Exception as e:
        print(f"  > Skipping: Error reading file. {e}")
        return

    # Use difflib to find differences
    diff = list(difflib.unified_diff(
        good_lines,
        bad_lines,
        fromfile=f"GOOD/{file_rel_path}",
        tofile=f"BAD/{file_rel_path}",
        n=2  # Number of context lines
    ))

    if not diff:
        print("  ✅ OK: XML content is identical (this is rare).")
    else:
        print("  (Showing differences. '-' is from GOOD, '+' is from BAD)")
        # Print first 50 lines of diff to avoid flooding
        for line in diff[:50]:
            print(f"  {line.strip()}")
        if len(diff) > 50:
            print(f"  ...and {len(diff) - 50} more differences.")

def main():
    print("Starting deep file structure analysis...")

    # 1. Unzip both files
    if not unzip_file(GOOD_FILE, GOOD_DIR) or not unzip_file(BAD_FILE, BAD_DIR):
        print("\nAnalysis halted due to unzipping error.")
        return

    # 2. Get file lists
    good_file_set = get_all_files(GOOD_DIR)
    bad_file_set = get_all_files(BAD_DIR)

    # 3. Compare lists
    compare_file_lists(good_file_set, bad_file_set)

    # 4. Compare properties of common files
    common_files = good_file_set.intersection(bad_file_set)
    compare_file_properties(GOOD_DIR, BAD_DIR, common_files)

    # 5. Diff key structural files
    # These are the most likely culprits for structural corruption
    key_files_to_diff = [
        "xl/workbook.xml",       # Defines sheets, hidden sheets, defined names
        "xl/styles.xml",         # Defines all cell formatting (COMMON CULPRIT)
        "[Content_Types].xml"    # Defines all parts of the package
    ]

    for f in key_files_to_diff:
        compare_file_content(GOOD_DIR, BAD_DIR, f)

    print("\n--- Analysis Complete ---")

    # Clean up
    try:
        shutil.rmtree(GOOD_DIR)
        shutil.rmtree(BAD_DIR)
        print(f"Cleaned up temp directories: {GOOD_DIR}, {BAD_DIR}")
    except Exception as e:
        print(f"Warning: Could not clean up temp directories. {e}")


if __name__ == "__main__":
    main()


Starting deep file structure analysis...

Extracting good.xlsx to good_file_unzipped...
  > Success.

Extracting FIXED_SURGICALLY.xlsx to bad_file_unzipped...
  > Success.

--- 1. Comparing File Lists ---
  ✅ OK: File lists match perfectly.

--- 2. Comparing File Sizes ---
  (Looking for major size differences, e.g., in styles.xml)
  ✅ OK: No significant file size differences found.

--- 3. Content Diff for: xl/workbook.xml ---
  (Showing differences. '-' is from GOOD, '+' is from BAD)
  --- GOOD/xl/workbook.xml
  +++ BAD/xl/workbook.xml
  @@ -1,2 +1,2 @@
  <?xml version="1.0" encoding="UTF-8" standalone="yes"?>
  -<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships" xmlns:mc="http://schemas.openxmlformats.org/markup-compatibility/2006" mc:Ignorable="x15 xr xr6 xr10 xr2" xmlns:x15="http://schemas.microsoft.com/office/spreadsheetml/2010/11/main" xmlns:xr="http://schemas.microsoft.com/off

# Attempting to fix metadata issue

In [ ]:
import zipfile
import shutil

# --- CONFIGURATION ---
BAD_FILE = "bad.xlsx"
GOOD_FILE = "good.xlsx" # Not strictly needed, but good to have for naming
OUTPUT_FILE = "FIXED_SURGICALLY.xlsx"

FILE_1 = "customXml/item1.xml"
FILE_2 = "customXml/item3.xml"
# --- END CONFIGURATION ---

def main():
    print(f"Starting surgical repair of {BAD_FILE}...")

    try:
        # 1. Read the contents of the two swapped files from the bad file
        with zipfile.ZipFile(BAD_FILE, 'r') as zin:
            print(f"  > Reading XML content from {BAD_FILE}...")

            # Read the file that *should* be small (but is big)
            # In your log: BAD item1.xml is 219 bytes. This is the small content.
            small_content = zin.read(FILE_1)

            # Read the file that *should* be big (but is small)
            # In your log: BAD item3.xml is 47620 bytes. This is the big content.
            large_content = zin.read(FILE_2)

        print(f"    - Read {len(small_content)} bytes from {FILE_1} (will move to {FILE_2})")
        print(f"    - Read {len(large_content)} bytes from {FILE_2} (will move to {FILE_1})")

        # 2. Create the new, fixed ZIP (XLSX) file
        # We must read from the source and write to a new destination
        with zipfile.ZipFile(BAD_FILE, 'r') as zin:
            with zipfile.ZipFile(OUTPUT_FILE, 'w', compression=zipfile.ZIP_DEFLATED) as zout:

                # Iterate over every file in the original ZIP
                for item in zin.infolist():

                    if item.filename == FILE_1:
                        # This is FILE_1. Write the LARGE content here instead.
                        # This reverses the swap.
                        zout.writestr(item, large_content)

                    elif item.filename == FILE_2:
                        # This is FILE_2. Write the SMALL content here instead.
                        # This completes the swap.
                        zout.writestr(item, small_content)

                    else:
                        # This is any other file (like your data, styles, etc.)
                        # Copy it over byte-for-byte, completely untouched.
                        zout.writestr(item, zin.read(item.filename))

        print(f"\n--- ✅ Success! ---")
        print(f"Surgical swap complete. Repaired file saved as: {OUTPUT_FILE}")

    except FileNotFoundError as e:
        print(f"❌ ERROR: File not found. Make sure '{e.filename}' is in the same directory.")
    except KeyError as e:
        print(f"❌ ERROR: A required XML file is missing from the zip: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    main()


Starting surgical repair of bad.xlsx...
  > Reading XML content from bad.xlsx...
    - Read 219 bytes from customXml/item1.xml (will move to customXml/item3.xml)
    - Read 47620 bytes from customXml/item3.xml (will move to customXml/item1.xml)

--- ✅ Success! ---
Surgical swap complete. Repaired file saved as: FIXED_SURGICALLY.xlsx
